# 🎯 Fine-Tune Ekman Class Weighting (Modular & Checkpoint Edition)

สมุดงานนี้ได้รับการออกแบบมาเพื่อให้ปรับแต่งและเทรนโมเดลได้สะดวก ปลอดภัย และยืดหยุ่นสูงสุด:

### 🌟 จุดเด่นหลักของเวอร์ชันนี้:
1. **📁 ปรับเปลี่ยน Path ที่เซฟโมเดลได้ง่ายมาก**: กำหนดไว้ในกล่องคอนฟิกด้านบน สามารถเปลี่ยนโฟลเดอร์ปลายทางได้ทันที
2. **🎛️ แยกเซลล์การเทรนทีละโมเดลอิสระ (Model 1, Model 2, Model 3)**: คุณสามารถเลือกคลิกรันเฉพาะโมเดลที่ต้องการเทรนได้ ไม่ต้องรันต่อกันแบบบังคับ
3. **⚙️ กำหนด Hyperparameters แต่ละโมเดลได้อิสระ**: ตั้งค่า `LR`, `Epochs`, `Batch Size`, และ `Patience` ของแต่ละโมเดลแยกกันได้ชัดเจน
4. **📉 มีระบบ Learning Rate Scheduler**: รองรับทั้ง `cosine`, `linear` (พร้อม warmup) และ `reduce_on_plateau`
5. **💾 ระบบ Best & Epoch Checkpoint + Rollback ย้อนกลับ**: 
   - เซฟ Checkpoint ทุกๆ Epoch พร้อมบันทึก Loss และ Metadata
   - เซฟ `best_model` อัตโนมัติเมื่อได้ Validation Loss ต่ำสุด
   - หากโมเดล Epoch ล่าสุดเกิด Overfit มีคำสั่ง **Rollback** ย้อนกลับไปใช้ Checkpoint ที่ดีที่สุดได้ใน 1 บรรทัด
6. **📊 ระบบ Logging ละเอียด**: บันทึก Log ประวัติการเทรน (Loss, LR, Epoch) ลงไฟล์ JSON เพื่อตรวจสอบย้อนหลังได้ตลอดเวลา
7. **⚡ Zero VRAM Overflow**: เคลียร์ VRAM คืนหน่วยความจำ GPU ทันทีหลังเทรนแต่ละโมเดลเสร็จ

## ⚙️ Step 1: ปรับแต่งโฟลเดอร์เซฟโมเดล & โฟลเดอร์ Logs (แก้ไขได้ที่นี่)

In [3]:
import os

# ============================================================
# 📁 กำหนดที่เซฟโมเดล และ Logs ได้ง่ายๆ ตรงนี้เลยครับ
# ============================================================
BASE_SAVE_DIR = "saved_models_ekman_finetune"
LOG_SAVE_DIR  = "training_logs_ekman"
RESULTS_DIR   = "results"

# เช็ค path อัตโนมัติ (กรณีรันจากโฟลเดอร์ notebooks/ หรือ root)
def resolve_dir(dir_name):
    if not os.path.exists(dir_name) and os.path.exists(os.path.join("..", dir_name)):
        return os.path.join("..", dir_name)
    elif not os.path.exists(dir_name) and os.path.exists("../saved_models"):
        return os.path.join("..", dir_name)
    return dir_name

BASE_SAVE_DIR = resolve_dir(BASE_SAVE_DIR)
LOG_SAVE_DIR  = resolve_dir(LOG_SAVE_DIR)
RESULTS_DIR   = resolve_dir(RESULTS_DIR)

os.makedirs(BASE_SAVE_DIR, exist_ok=True)
os.makedirs(LOG_SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"📁 โฟลเดอร์สำหรับเซฟโมเดลและ Checkpoint : {os.path.abspath(BASE_SAVE_DIR)}")
print(f"📊 โฟลเดอร์สำหรับเซฟ Training Logs    : {os.path.abspath(LOG_SAVE_DIR)}")
print(f"📝 โฟลเดอร์สำหรับเซฟผลการประเมิน       : {os.path.abspath(RESULTS_DIR)}")

📁 โฟลเดอร์สำหรับเซฟโมเดลและ Checkpoint : /home/apiwit/NLP_project/saved_models_ekman_finetune
📊 โฟลเดอร์สำหรับเซฟ Training Logs    : /home/apiwit/NLP_project/training_logs_ekman
📝 โฟลเดอร์สำหรับเซฟผลการประเมิน       : /home/apiwit/NLP_project/results


## 📦 Step 2: Import Libraries & กำหนดค่า Device (GPU/CPU)

In [4]:
import ast
import json
import joblib
import gc
import sys
import shutil
import numpy as np
import pandas as pd
from datetime import datetime
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import (
    CosineAnnealingLR, 
    ReduceLROnPlateau
)
from torch.nn import BCEWithLogitsLoss
from torch.amp import autocast, GradScaler
from transformers import ( 
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
    get_linear_schedule_with_warmup
    )
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, hamming_loss, f1_score
from tqdm.auto import tqdm

# เชื่อมต่อ eval_logger ใน src/ สำหรับบันทึก leaderboard
for path in ['..', '.']:
    if path not in sys.path and os.path.exists(os.path.join(path, 'src', 'eval_logger.py')):
        sys.path.append(path)
        break

try:
    from src.eval_logger import save_eval_report
    print("✅ เชื่อมต่อโมดูล save_eval_report เรียบร้อย")
except ImportError:
    save_eval_report = None
    print("⚠️ ไม่พบโมดูล src.eval_logger (ผลจะแสดงบนหน้าจอแต่ไม่เขียนลง leaderboard อัตโนมัติ)")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Running on device: {device}")
if torch.cuda.is_available():
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM Capacity: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

✅ เชื่อมต่อโมดูล save_eval_report เรียบร้อย
🚀 Running on device: cuda
   GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU
   VRAM Capacity: 8.00 GB


## 📊 Step 3: โหลดข้อมูล & ทำ Ekman 7-Class Mapping

In [5]:
TRAIN_PATH = '../goemotions/data/go_emotions_train_with_translation.csv'
VAL_PATH   = '../goemotions/data/go_emotions_validation_with_translation.csv'
TEST_PATH  = '../goemotions/data/go_emotions_test_with_translation.csv'

if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = 'goemotions/data/go_emotions_train_with_translation.csv'
    VAL_PATH   = 'goemotions/data/go_emotions_validation_with_translation.csv'
    TEST_PATH  = 'goemotions/data/go_emotions_test_with_translation.csv'

df_train = pd.read_csv(TRAIN_PATH).dropna(subset=['text_th']).reset_index(drop=True)
df_val   = pd.read_csv(VAL_PATH).dropna(subset=['text_th']).reset_index(drop=True)
df_test  = pd.read_csv(TEST_PATH).dropna(subset=['text_th']).reset_index(drop=True)

print(f'จำนวนข้อมูล Train ภาษาไทย      : {len(df_train):,} แถว')
print(f'จำนวนข้อมูล Validation ภาษาไทย : {len(df_val):,} แถว')
print(f'จำนวนข้อมูล Test ภาษาไทย       : {len(df_test):,} แถว')

ORIGINAL_LABELS = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval',
    'caring', 'confusion', 'curiosity', 'desire', 'disappointment',
    'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear',
    'gratitude', 'grief', 'joy', 'love', 'nervousness',
    'optimism', 'pride', 'realization', 'relief', 'remorse',
    'sadness', 'surprise', 'neutral'
]

EKMAN_MAPPING = {
    'Joy': [
        'admiration', 'amusement', 'approval', 'caring', 'desire',
        'excitement', 'gratitude', 'joy', 'love', 'optimism', 'pride', 'relief'
    ],
    'Anger': ['anger', 'annoyance', 'disapproval'],
    'Sadness': ['disappointment', 'embarrassment', 'grief', 'remorse', 'sadness'],
    'Surprise': ['confusion', 'curiosity', 'realization', 'surprise'],
    'Fear': ['fear', 'nervousness'],
    'Disgust': ['disgust'],
    'Neutral': ['neutral']
}

EKMAN_LABELS = list(EKMAN_MAPPING.keys())
NUM_LABELS = len(EKMAN_LABELS)

ekman_label_to_id = {lbl: i for i, lbl in enumerate(EKMAN_LABELS)}
orig_id_to_ekman_id = {}
for orig_id, orig_lbl in enumerate(ORIGINAL_LABELS):
    for ekman_lbl, orig_list in EKMAN_MAPPING.items():
        if orig_lbl in orig_list:
            orig_id_to_ekman_id[orig_id] = ekman_label_to_id[ekman_lbl]
            break

def map_labels(labels_raw):
    mapped = set()
    for lbl in labels_raw:
        if isinstance(lbl, int) and lbl in orig_id_to_ekman_id:
            mapped.add(orig_id_to_ekman_id[lbl])
        elif isinstance(lbl, str):
            for ek_lbl, orig_list in EKMAN_MAPPING.items():
                if lbl in orig_list:
                    mapped.add(ekman_label_to_id[ek_lbl])
    return list(mapped)

for df in [df_train, df_val, df_test]:
    df['labels_list'] = df['labels'].apply(ast.literal_eval)
    df['ekman_labels'] = df['labels_list'].apply(map_labels)

mlb = MultiLabelBinarizer(classes=list(range(NUM_LABELS)))
Y_train = mlb.fit_transform(df_train['ekman_labels'])
X_train = df_train['text_th'].astype(str).tolist()

Y_val = mlb.transform(df_val['ekman_labels'])
X_val = df_val['text_th'].astype(str).tolist()

Y_test = mlb.transform(df_test['ekman_labels'])
X_test = df_test['text_th'].astype(str).tolist()

print(f'✅ แมปปิ้ง Ekman 7 สำเร็จ! Train Shape: {Y_train.shape} | Val Shape: {Y_val.shape} | Test Shape: {Y_test.shape}')

จำนวนข้อมูล Train ภาษาไทย      : 43,410 แถว
จำนวนข้อมูล Validation ภาษาไทย : 5,426 แถว
จำนวนข้อมูล Test ภาษาไทย       : 5,131 แถว
✅ แมปปิ้ง Ekman 7 สำเร็จ! Train Shape: (43410, 7) | Val Shape: (5426, 7) | Test Shape: (5131, 7)


## ⚖️ Step 4: Level 1 Class Weighting (Boosted pos_weight)

In [6]:
pos_counts = Y_train.sum(axis=0)
neg_counts = len(Y_train) - pos_counts
base_weight = neg_counts / (pos_counts + 1e-6)

# Boost Factor สำหรับคลาสชนกลุ่มน้อย (Minority Classes):
# [0] Joy, [1] Anger, [2] Sadness, [3] Surprise, [4] Fear, [5] Disgust, [6] Neutral
BOOST_FACTOR = np.array([
    1.0,   # Joy
    1.0,   # Anger
    2.5,   # Sadness  (boost 2.5x)
    1.0,   # Surprise
    3.0,   # Fear     (boost 3.0x)
    3.0,   # Disgust  (boost 3.0x)
    1.0,   # Neutral
])

boosted_weight = base_weight * BOOST_FACTOR
pos_weight_tensor = torch.FloatTensor(boosted_weight).to(device)

print('⚖️ Boosted pos_weight สำหรับ Loss Function:')
for i, lbl in enumerate(EKMAN_LABELS):
    boost_tag = f'>> BOOSTED x{BOOST_FACTOR[i]}' if BOOST_FACTOR[i] > 1.0 else ''
    print(f'  [{i}] {lbl:<10} | pos={int(pos_counts[i]):>5} | pos_weight={boosted_weight[i]:.2f} {boost_tag}')

⚖️ Boosted pos_weight สำหรับ Loss Function:
  [0] Joy        | pos=17410 | pos_weight=1.49 
  [1] Anger      | pos= 5579 | pos_weight=6.78 
  [2] Sadness    | pos= 3263 | pos_weight=30.76 >> BOOSTED x2.5
  [3] Surprise   | pos= 5367 | pos_weight=7.09 
  [4] Fear       | pos=  726 | pos_weight=176.38 >> BOOSTED x3.0
  [5] Disgust    | pos=  793 | pos_weight=161.22 >> BOOSTED x3.0
  [6] Neutral    | pos=14219 | pos_weight=2.05 


## 🛠️ Step 5: Dataset, Checkpoint Manager & Logging Utilities

In [7]:
class ThaiEmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'labels': torch.FloatTensor(self.labels[idx])
        }

class CheckpointManager:
    """
    ระบบจัดการ Checkpoint ต่อโมเดล:
    - บันทึก checkpoint ทุก Epoch (เช่น checkpoint-epoch-1, 2, ...)
    - บันทึก best_model อัตโนมัติเมื่อ Val Loss ต่ำสุด
    - เก็บประวัติและ metadata (Val Loss, Train Loss, LR) ไว้ใน json
    - สามารถสั่งย้อนกลับ (Rollback) หรือก๊อปปี้ Checkpoint ที่ต้องการมาใช้งานได้ทันที
    """
    def __init__(self, model_key, base_dir=BASE_SAVE_DIR):
        self.model_key = model_key
        self.model_dir = os.path.join(base_dir, f"{model_key}_model")
        self.tok_dir   = os.path.join(base_dir, f"{model_key}_tokenizer")
        self.ckpt_dir  = os.path.join(base_dir, f"{model_key}_checkpoints")
        os.makedirs(self.ckpt_dir, exist_ok=True)
        
        self.meta_file = os.path.join(self.ckpt_dir, "checkpoints_metadata.json")
        self.history = self._load_meta()
        self.best_val_loss = self._get_best_val_loss()

    def _load_meta(self):
        if os.path.exists(self.meta_file):
            try:
                with open(self.meta_file, 'r', encoding='utf-8') as f:
                    return json.load(f)
            except Exception:
                return []
        return []

    def _save_meta(self):
        with open(self.meta_file, 'w', encoding='utf-8') as f:
            json.dump(self.history, f, ensure_ascii=False, indent=2)

    def _get_best_val_loss(self):
        if not self.history:
            return float('inf')
        return min(item['val_loss'] for item in self.history)

    def save_checkpoint(self, model, tokenizer, epoch, train_loss, val_loss, current_lr):
        epoch_name = f"epoch_{epoch}"
        save_path = os.path.join(self.ckpt_dir, epoch_name)
        
        # บันทึก Checkpoint ของ Epoch ปัจจุบัน
        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        
        is_best = val_loss < self.best_val_loss
        if is_best:
            self.best_val_loss = val_loss
            # อัปเดตไปยังโมเดลหลักที่พร้อมใช้งาน
            model.save_pretrained(self.model_dir)
            tokenizer.save_pretrained(self.tok_dir)
        
        record = {
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "lr": float(current_lr),
            "is_best": bool(is_best),
            "checkpoint_path": save_path,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        self.history.append(record)
        self._save_meta()
        return is_best

    def list_checkpoints(self):
        print(f"\n📑 รายการ Checkpoints ทั้งหมดของ [{self.model_key}]:")
        if not self.history:
            print("   (ยังไม่มีประวัติ Checkpoint)")
            return
        for h in self.history:
            star = "🏆 [BEST]" if h.get('is_best') else "  " 
            print(f"   {star} Epoch {h['epoch']} | Val Loss: {h['val_loss']:.4f} | Train Loss: {h['train_loss']:.4f} | LR: {h['lr']:.2e} | เวลา: {h['timestamp']}")

    def rollback_to_epoch(self, target_epoch):
        """ย้อนคืนโมเดลหลักกลับไปใช้ Checkpoint ของ Epoch ที่เลือก"""
        source_ckpt = os.path.join(self.ckpt_dir, f"epoch_{target_epoch}")
        if not os.path.exists(source_ckpt):
            print(f"❌ ไม่พบ Checkpoint ของ Epoch {target_epoch} ที่: {source_ckpt}")
            return False
        
        print(f"⏪ กำลัง Rollback โมเดลหลัก [{self.model_key}] ไปใช้ Epoch {target_epoch}...")
        if os.path.exists(self.model_dir):
            shutil.rmtree(self.model_dir)
        shutil.copytree(source_ckpt, self.model_dir)
        print(f"✅ Rollback สำเร็จ! โมเดลหลักที่ {self.model_dir} กลายเป็นเวอร์ชัน Epoch {target_epoch} แล้ว")
        return True

    def rollback_to_best(self):
        """ย้อนคืนโมเดลหลักกลับไปใช้ Best Checkpoint ที่มี Val Loss ต่ำสุด"""
        if not self.history:
            print("❌ ไม่มีประวัติ Checkpoint")
            return False
        best_record = min(self.history, key=lambda x: x['val_loss'])
        return self.rollback_to_epoch(best_record['epoch'])


def save_run_log(model_key, config, history, log_dir=LOG_SAVE_DIR):
    """บันทึก log สรุปการเทรนแต่ละ run ลงไฟล์ json"""
    os.makedirs(log_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = os.path.join(log_dir, f"{model_key}_run_{timestamp}.json")
    log_data = {
        "model_key": model_key,
        "timestamp": timestamp,
        "config": config,
        "history": history
    }
    with open(log_file, 'w', encoding='utf-8') as f:
        json.dump(log_data, f, ensure_ascii=False, indent=2)
    print(f"📝 บันทึก Log การเทรนรอบนี้ไว้ที่: {log_file}")

print("✅ เครื่องมือ CheckpointManager & Logger พร้อมทำงาน!")

✅ เครื่องมือ CheckpointManager & Logger พร้อมทำงาน!


## ⚙️ Step 6: ฟังก์ชันการเทรนอเนกประสงค์ (รองรับ LR Scheduler, Checkpoint & Overfit Alert)

In [8]:
def train_custom_model(
    model_key,
    pretrained_name,
    use_fast=True,
    epochs=10,
    lr=5e-5,
    batch_size=16,
    scheduler_type='cosine',    # 'cosine', 'linear', 'plateau', หรือ 'none'
    warmup_ratio=0.1,           # สัดส่วน warmup step
    early_stopping_patience=2,  # 0 คือปิด early stopping, เช่น 2 คือหยุดถ้า val loss ไม่ลงติดกัน 2 epoch
    load_existing=False         # True = โหลดโมเดลเดิมมาเทรนต่อ, False = เริ่มจาก pretrained weights
):
    """
    ฟังก์ชันเทรนแบบปรับแต่งละเอียด:
    - ปรับ LR / Epochs / Scheduler ได้อิสระ
    - บันทึก Checkpoint ทุก Epoch และแยกเก็บ Best Model
    - มีระบบเตือนเมื่อเริ่ม Overfit
    - Zero VRAM: เคลียร์หน่วยความจำอัตโนมัติเมื่อสิ้นสุด
    """
    ckpt_mgr = CheckpointManager(model_key=model_key, base_dir=BASE_SAVE_DIR)
    
    print(f"\n{'='*70}")
    mode_str = "🔄 โหลดโมเดลเดิมมาเทรนต่อ (Continue)" if load_existing else "🚀 เริ่มเทรนใหม่ (Fresh Train)"
    print(f"{mode_str}: [{model_key}] ({pretrained_name})")
    print(f"   ⚙️ Epochs: {epochs} | LR: {lr:.2e} | Batch: {batch_size} | Scheduler: {scheduler_type}")
    print(f"{'='*70}")
    
    # เลือกว่าจะโหลดจาก path เดิมหรือจาก pretrained hub
    if load_existing and os.path.exists(ckpt_mgr.model_dir):
        print(f"📂 กำลังโหลดโมเดลเดิมจาก: {ckpt_mgr.model_dir}")
        tok = AutoTokenizer.from_pretrained(ckpt_mgr.tok_dir, use_fast=use_fast)
        model = AutoModelForSequenceClassification.from_pretrained(
            ckpt_mgr.model_dir, num_labels=NUM_LABELS, problem_type='multi_label_classification'
        ).to(device)
    else:
        tok = AutoTokenizer.from_pretrained(pretrained_name, use_fast=use_fast)
        model = AutoModelForSequenceClassification.from_pretrained(
            pretrained_name, num_labels=NUM_LABELS, problem_type='multi_label_classification'
        ).to(device)
        
    train_loader = DataLoader(ThaiEmotionDataset(X_train, Y_train, tok), batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(ThaiEmotionDataset(X_val,   Y_val,   tok), batch_size=batch_size, shuffle=False)
    
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    criterion = BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    scaler = GradScaler('cuda' if torch.cuda.is_available() else 'cpu')
    
    total_steps = len(train_loader) * epochs
    warmup_steps = int(total_steps * warmup_ratio)
    
    # เลือกระบบ LR Scheduler
    scheduler = None
    if scheduler_type == 'cosine':
        scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    elif scheduler_type == 'linear':
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    elif scheduler_type == 'plateau':
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1, min_lr=1e-7)
    
    history = []
    patience_counter = 0
    
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]")
        for b in pbar:
            optimizer.zero_grad()
            with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
                out = model(b['input_ids'].to(device), attention_mask=b['attention_mask'].to(device))
                loss = criterion(out.logits, b['labels'].to(device))
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            if scheduler is not None and scheduler_type in ['cosine', 'linear']:
                scheduler.step()
                
            train_loss += loss.item()
            current_step_lr = optimizer.param_groups[0]['lr']
            pbar.set_postfix({'loss': f"{loss.item():.4f}", 'lr': f"{current_step_lr:.2e}"})
            
        # Phase Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for b in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [Val]", leave=False):
                with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
                    out = model(b['input_ids'].to(device), attention_mask=b['attention_mask'].to(device))
                    val_loss += criterion(out.logits, b['labels'].to(device)).item()
                    
        avg_train = train_loss / len(train_loader)
        avg_val   = val_loss / len(val_loader)
        current_epoch_lr = optimizer.param_groups[0]['lr']
        
        # ถ้าใช้ Plateau ให้ step ตอนจบ epoch ตาม val loss
        if scheduler is not None and scheduler_type == 'plateau':
            scheduler.step(avg_val)
            
        # เซฟ Checkpoint
        is_best = ckpt_mgr.save_checkpoint(
            model=model,
            tokenizer=tok,
            epoch=epoch,
            train_loss=avg_train,
            val_loss=avg_val,
            current_lr=current_epoch_lr
        )
        
        history.append({
            "epoch": epoch,
            "train_loss": avg_train,
            "val_loss": avg_val,
            "lr": current_epoch_lr,
            "is_best": is_best
        })
        
        print(f"  ✨ Epoch {epoch}/{epochs} -> Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f} | LR: {current_epoch_lr:.2e}")
        if is_best:
            print(f"     🏆 [Best Model] พบ Val Loss ต่ำสุดใหม่ ({avg_val:.4f}) -> อัปเดตโมเดลหลักพร้อมใช้งาน!")
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"     ⚠️ [Overfit Alert] Val Loss ไม่ลดลงจากจุดดีที่สุด ({ckpt_mgr.best_val_loss:.4f}) -> จุดนี้ถูกเก็บเป็น Checkpoint ไว้แล้ว")
            
        # ระบบ Early Stopping
        if early_stopping_patience > 0 and patience_counter >= early_stopping_patience:
            print(f"\n🛑 Early Stopping ทำงาน! (Val Loss ไม่ลดลงติดต่อกัน {early_stopping_patience} Epochs)")
            print(f"💡 แนะนำ: คุณสามารถสั่ง ckpt_mgr.rollback_to_best() เพื่อใช้โมเดลตัวที่ดีที่สุดได้เลย")
            break
            
    # เคลียร์ VRAM คืน GPU
    model.to('cpu')
    del optimizer, train_loader, val_loader, scaler
    if scheduler is not None:
        del scheduler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    
    # บันทึก Run Log
    run_config = {
        "epochs": epochs,
        "lr": lr,
        "batch_size": batch_size,
        "scheduler_type": scheduler_type,
        "warmup_ratio": warmup_ratio,
        "load_existing": load_existing
    }
    save_run_log(model_key, run_config, history)
    
    print(f"\n🎉 เสร็จสิ้นการทำงานของโมเดล [{model_key}]!")
    ckpt_mgr.list_checkpoints()
    return ckpt_mgr

## 🚀 โมเดล 1: WangchanBERTa

> **คุณสามารถปรับตั้งค่า `EPOCHS`, `LR`, `BATCH_SIZE`, `SCHEDULER` ได้ที่เซลล์นี้ แล้วกดรันเฉพาะเซลล์นี้ได้เลยครับ**

In [8]:
# ============================================================
# 🎛️ ปรับแต่ง Hyperparameters สำหรับ โมเดล 1 (WangchanBERTa)
# ============================================================
M1_KEY             = 'wangchan'
M1_PRETRAINED_NAME = 'airesearch/wangchanberta-base-att-spm-uncased'
M1_USE_FAST        = False

M1_EPOCHS          = 10           # 🔧 จำนวน Epochs (เช่น 3 หรือ 4)
M1_LR              = 5e-5        # 🔧 Learning Rate (เช่น 2e-5 หรือ 1.5e-5)
M1_BATCH_SIZE      = 16          # 🔧 Batch Size (16 กำลังดีสำหรับ VRAM 8GB)
M1_SCHEDULER       = 'cosine'    # 🔧 'cosine', 'linear', 'plateau', หรือ 'none'
M1_PATIENCE        = 2           # 🔧 0 คือไม่หยุดอัตโนมัติ, หรือใส่ 2 เพื่อหยุดเมื่อ overfit
M1_LOAD_EXISTING   = False       # 🔧 True = โหลดโมเดลเดิมมาเทรนต่อ, False = เริ่มเทรนใหม่

# 🚀 สั่งรันเทรนเฉพาะ WangchanBERTa
ckpt_wangchan = train_custom_model(
    model_key=M1_KEY,
    pretrained_name=M1_PRETRAINED_NAME,
    use_fast=M1_USE_FAST,
    epochs=M1_EPOCHS,
    lr=M1_LR,
    batch_size=M1_BATCH_SIZE,
    scheduler_type=M1_SCHEDULER,
    early_stopping_patience=M1_PATIENCE,
    load_existing=M1_LOAD_EXISTING
)


🚀 เริ่มเทรนใหม่ (Fresh Train): [wangchan] (airesearch/wangchanberta-base-att-spm-uncased)
   ⚙️ Epochs: 10 | LR: 5.00e-05 | Batch: 16 | Scheduler: cosine


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] CamembertForSequenceClassification LOAD REPORT from: airesearch/wangchanberta-base-att-spm-uncased
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/10 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

/tmp/ipykernel_11440/1814583397.py:78: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 1/10 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

... (12 outputs left out)

Epoch 4/10 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 4/10 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✨ Epoch 4/10 -> Train Loss: 1.5213 | Val Loss: 1.5156 | LR: 3.75e-05
     ⚠️ [Overfit Alert] Val Loss ไม่ลดลงจากจุดดีที่สุด (0.8609) -> จุดนี้ถูกเก็บเป็น Checkpoint ไว้แล้ว

🛑 Early Stopping ทำงาน! (Val Loss ไม่ลดลงติดต่อกัน 2 Epochs)
💡 แนะนำ: คุณสามารถสั่ง ckpt_mgr.rollback_to_best() เพื่อใช้โมเดลตัวที่ดีที่สุดได้เลย
📝 บันทึก Log การเทรนรอบนี้ไว้ที่: ../training_logs_ekman/wangchan_run_20261025_152227.json

🎉 เสร็จสิ้นการทำงานของโมเดล [wangchan]!

📑 รายการ Checkpoints ทั้งหมดของ [wangchan]:
   🏆 [BEST] Epoch 1 | Val Loss: 0.9442 | Train Loss: 1.2808 | LR: 5.00e-05 | เวลา: 2026-10-25 15:07:27
   🏆 [BEST] Epoch 2 | Val Loss: 0.8609 | Train Loss: 0.9643 | LR: 4.85e-05 | เวลา: 2026-10-25 15:12:26
      Epoch 3 | Val Loss: 1.5348 | Train Loss: 1.0541 | LR: 4.42e-05 | เวลา: 2026-10-25 15:17:26
      Epoch 4 | Val Loss: 1.5156 | Train Loss: 1.5213 | LR: 3.75e-05 | เวลา: 2026-10-25 15:22:27


In [10]:
ckpt_wangchan.rollback_to_best()

⏪ กำลัง Rollback โมเดลหลัก [wangchan] ไปใช้ Epoch 2...
✅ Rollback สำเร็จ! โมเดลหลักที่ ../saved_models_ekman_finetune/wangchan_model กลายเป็นเวอร์ชัน Epoch 2 แล้ว


True

เริ่มรอบสอง ลด LR ลง 

In [11]:
# ============================================================
# 🎛️ ปรับแต่ง Hyperparameters สำหรับ โมเดล 1 (WangchanBERTa)
# ============================================================
M1_KEY             = 'wangchan'
M1_PRETRAINED_NAME = 'airesearch/wangchanberta-base-att-spm-uncased'
M1_USE_FAST        = False

M1_EPOCHS          = 5           # 🔧 จำนวน Epochs (เช่น 3 หรือ 4)
M1_LR              = 2e-5        # 🔧 Learning Rate (เช่น 2e-5 หรือ 1.5e-5)
M1_BATCH_SIZE      = 16          # 🔧 Batch Size (16 กำลังดีสำหรับ VRAM 8GB)
M1_SCHEDULER       = 'cosine'    # 🔧 'cosine', 'linear', 'plateau', หรือ 'none'
M1_PATIENCE        = 2           # 🔧 0 คือไม่หยุดอัตโนมัติ, หรือใส่ 2 เพื่อหยุดเมื่อ overfit
M1_LOAD_EXISTING   = True       # 🔧 True = โหลดโมเดลเดิมมาเทรนต่อ, False = เริ่มเทรนใหม่

# 🚀 สั่งรันเทรนเฉพาะ WangchanBERTa
ckpt_wangchan = train_custom_model(
    model_key=M1_KEY,
    pretrained_name=M1_PRETRAINED_NAME,
    use_fast=M1_USE_FAST,
    epochs=M1_EPOCHS,
    lr=M1_LR,
    batch_size=M1_BATCH_SIZE,
    scheduler_type=M1_SCHEDULER,
    early_stopping_patience=M1_PATIENCE,
    load_existing=M1_LOAD_EXISTING
)


🔄 โหลดโมเดลเดิมมาเทรนต่อ (Continue): [wangchan] (airesearch/wangchanberta-base-att-spm-uncased)
   ⚙️ Epochs: 5 | LR: 2.00e-05 | Batch: 16 | Scheduler: cosine
📂 กำลังโหลดโมเดลเดิมจาก: ../saved_models_ekman_finetune/wangchan_model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch 1/5 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 1/5 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (5 outputs left out)

Epoch 3/5 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 3/5 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✨ Epoch 3/5 -> Train Loss: 1.5172 | Val Loss: 1.5142 | LR: 8.26e-06
     ⚠️ [Overfit Alert] Val Loss ไม่ลดลงจากจุดดีที่สุด (0.8601) -> จุดนี้ถูกเก็บเป็น Checkpoint ไว้แล้ว

🛑 Early Stopping ทำงาน! (Val Loss ไม่ลดลงติดต่อกัน 2 Epochs)
💡 แนะนำ: คุณสามารถสั่ง ckpt_mgr.rollback_to_best() เพื่อใช้โมเดลตัวที่ดีที่สุดได้เลย
📝 บันทึก Log การเทรนรอบนี้ไว้ที่: ../training_logs_ekman/wangchan_run_20261025_155012.json

🎉 เสร็จสิ้นการทำงานของโมเดล [wangchan]!

📑 รายการ Checkpoints ทั้งหมดของ [wangchan]:
   🏆 [BEST] Epoch 1 | Val Loss: 0.9442 | Train Loss: 1.2808 | LR: 5.00e-05 | เวลา: 2026-10-25 15:07:27
   🏆 [BEST] Epoch 2 | Val Loss: 0.8609 | Train Loss: 0.9643 | LR: 4.85e-05 | เวลา: 2026-10-25 15:12:26
      Epoch 3 | Val Loss: 1.5348 | Train Loss: 1.0541 | LR: 4.42e-05 | เวลา: 2026-10-25 15:17:26
      Epoch 4 | Val Loss: 1.5156 | Train Loss: 1.5213 | LR: 3.75e-05 | เวลา: 2026-10-25 15:22:27
   🏆 [BEST] Epoch 1 | Val Loss: 0.8601 | Train Loss: 0.7586 | LR: 1.94e-05 | เวลา: 2026-10-25 15:40:07

In [12]:
ckpt_wangchan.rollback_to_best()

⏪ กำลัง Rollback โมเดลหลัก [wangchan] ไปใช้ Epoch 1...
✅ Rollback สำเร็จ! โมเดลหลักที่ ../saved_models_ekman_finetune/wangchan_model กลายเป็นเวอร์ชัน Epoch 1 แล้ว


True

## 🚀 โมเดล 2: mBERT (Multilingual BERT)

> **คุณสามารถปรับตั้งค่า `EPOCHS`, `LR`, `BATCH_SIZE`, `SCHEDULER` ได้ที่เซลล์นี้ แล้วกดรันเฉพาะเซลล์นี้ได้เลยครับ**

In [13]:
# ============================================================
# 🎛️ ปรับแต่ง Hyperparameters สำหรับ โมเดล 2 (mBERT)
# ============================================================
M2_KEY             = 'mbert'
M2_PRETRAINED_NAME = 'bert-base-multilingual-cased'
M2_USE_FAST        = True

M2_EPOCHS          = 10           # 🔧 แนะนำ 3-4 Epochs
M2_LR              = 5e-5        # 🔧 แนะนำ 2e-5 หรือ 1.5e-5
M2_BATCH_SIZE      = 16          # 🔧 Batch Size
M2_SCHEDULER       = 'cosine'    # 🔧 'cosine', 'linear', 'plateau', หรือ 'none'
M2_PATIENCE        = 2           # 🔧 หยุดทันทีถ้า val loss ไม่ลดลง 2 รอบติด
M2_LOAD_EXISTING   = False       # 🔧 True = เทรนต่อจากเดิม, False = เริ่มใหม่

# 🚀 สั่งรันเทรนเฉพาะ mBERT
ckpt_mbert = train_custom_model(
    model_key=M2_KEY,
    pretrained_name=M2_PRETRAINED_NAME,
    use_fast=M2_USE_FAST,
    epochs=M2_EPOCHS,
    lr=M2_LR,
    batch_size=M2_BATCH_SIZE,
    scheduler_type=M2_SCHEDULER,
    early_stopping_patience=M2_PATIENCE,
    load_existing=M2_LOAD_EXISTING
)


🚀 เริ่มเทรนใหม่ (Fresh Train): [mbert] (bert-base-multilingual-cased)
   ⚙️ Epochs: 10 | LR: 5.00e-05 | Batch: 16 | Scheduler: cosine


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/10 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

/tmp/ipykernel_11440/1814583397.py:78: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 1/10 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

... (7 outputs left out)

Epoch 3/10 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 3/10 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✨ Epoch 3/10 -> Train Loss: 1.5153 | Val Loss: 1.5356 | LR: 4.42e-05
     ⚠️ [Overfit Alert] Val Loss ไม่ลดลงจากจุดดีที่สุด (1.2240) -> จุดนี้ถูกเก็บเป็น Checkpoint ไว้แล้ว

🛑 Early Stopping ทำงาน! (Val Loss ไม่ลดลงติดต่อกัน 2 Epochs)
💡 แนะนำ: คุณสามารถสั่ง ckpt_mgr.rollback_to_best() เพื่อใช้โมเดลตัวที่ดีที่สุดได้เลย
📝 บันทึก Log การเทรนรอบนี้ไว้ที่: ../training_logs_ekman/mbert_run_20261025_161715.json

🎉 เสร็จสิ้นการทำงานของโมเดล [mbert]!

📑 รายการ Checkpoints ทั้งหมดของ [mbert]:
   🏆 [BEST] Epoch 1 | Val Loss: 1.2240 | Train Loss: 1.3781 | LR: 5.00e-05 | เวลา: 2026-10-25 16:04:27
      Epoch 2 | Val Loss: 1.5157 | Train Loss: 1.4698 | LR: 4.85e-05 | เวลา: 2026-10-25 16:10:51
      Epoch 3 | Val Loss: 1.5356 | Train Loss: 1.5153 | LR: 4.42e-05 | เวลา: 2026-10-25 16:17:14


In [14]:
ckpt_mbert.rollback_to_best()

⏪ กำลัง Rollback โมเดลหลัก [mbert] ไปใช้ Epoch 1...
✅ Rollback สำเร็จ! โมเดลหลักที่ ../saved_models_ekman_finetune/mbert_model กลายเป็นเวอร์ชัน Epoch 1 แล้ว


True

In [16]:
# ============================================================
# 🎛️ ปรับแต่ง Hyperparameters สำหรับ โมเดล 2 (mBERT)
# ============================================================
M2_KEY             = 'mbert'
M2_PRETRAINED_NAME = 'bert-base-multilingual-cased'
M2_USE_FAST        = True

M2_EPOCHS          = 5           # 🔧 แนะนำ 3-4 Epochs
M2_LR              = 3.5e-5        # 🔧 แนะนำ 2e-5 หรือ 1.5e-5
M2_BATCH_SIZE      = 16          # 🔧 Batch Size
M2_SCHEDULER       = 'linear'    # 🔧 'cosine', 'linear', 'plateau', หรือ 'none'
M2_PATIENCE        = 2           # 🔧 หยุดทันทีถ้า val loss ไม่ลดลง 2 รอบติด
M2_LOAD_EXISTING   = False       # 🔧 True = เทรนต่อจากเดิม, False = เริ่มใหม่

# 🚀 สั่งรันเทรนเฉพาะ mBERT
ckpt_mbert = train_custom_model(
    model_key=M2_KEY,
    pretrained_name=M2_PRETRAINED_NAME,
    use_fast=M2_USE_FAST,
    epochs=M2_EPOCHS,
    lr=M2_LR,
    batch_size=M2_BATCH_SIZE,
    scheduler_type=M2_SCHEDULER,
    early_stopping_patience=M2_PATIENCE,
    load_existing=M2_LOAD_EXISTING
)


🚀 เริ่มเทรนใหม่ (Fresh Train): [mbert] (bert-base-multilingual-cased)
   ⚙️ Epochs: 5 | LR: 3.50e-05 | Batch: 16 | Scheduler: linear


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/5 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

/tmp/ipykernel_11440/1814583397.py:78: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 1/5 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

... (17 outputs left out)

Epoch 5/5 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 5/5 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✨ Epoch 5/5 -> Train Loss: 0.8374 | Val Loss: 1.0749 | LR: 0.00e+00
     ⚠️ [Overfit Alert] Val Loss ไม่ลดลงจากจุดดีที่สุด (0.9981) -> จุดนี้ถูกเก็บเป็น Checkpoint ไว้แล้ว

🛑 Early Stopping ทำงาน! (Val Loss ไม่ลดลงติดต่อกัน 2 Epochs)
💡 แนะนำ: คุณสามารถสั่ง ckpt_mgr.rollback_to_best() เพื่อใช้โมเดลตัวที่ดีที่สุดได้เลย
📝 บันทึก Log การเทรนรอบนี้ไว้ที่: ../training_logs_ekman/mbert_run_20261025_165341.json

🎉 เสร็จสิ้นการทำงานของโมเดล [mbert]!

📑 รายการ Checkpoints ทั้งหมดของ [mbert]:
   🏆 [BEST] Epoch 1 | Val Loss: 1.2240 | Train Loss: 1.3781 | LR: 5.00e-05 | เวลา: 2026-10-25 16:04:27
      Epoch 2 | Val Loss: 1.5157 | Train Loss: 1.4698 | LR: 4.85e-05 | เวลา: 2026-10-25 16:10:51
      Epoch 3 | Val Loss: 1.5356 | Train Loss: 1.5153 | LR: 4.42e-05 | เวลา: 2026-10-25 16:17:14
   🏆 [BEST] Epoch 1 | Val Loss: 1.1714 | Train Loss: 1.4282 | LR: 3.11e-05 | เวลา: 2026-10-25 16:28:03
   🏆 [BEST] Epoch 2 | Val Loss: 1.0871 | Train Loss: 1.1875 | LR: 2.33e-05 | เวลา: 2026-10-25 16:34:27
   🏆 [BE

In [17]:
ckpt_mbert.rollback_to_best()

⏪ กำลัง Rollback โมเดลหลัก [mbert] ไปใช้ Epoch 3...
✅ Rollback สำเร็จ! โมเดลหลักที่ ../saved_models_ekman_finetune/mbert_model กลายเป็นเวอร์ชัน Epoch 3 แล้ว


True

In [15]:
# ============================================================
# 🎛️ ปรับแต่ง Hyperparameters สำหรับ โมเดล 2 (mBERT)
# ============================================================
M2_KEY             = 'mbert'
M2_PRETRAINED_NAME = 'bert-base-multilingual-cased'
M2_USE_FAST        = True

M2_EPOCHS          = 10           # 🔧 แนะนำ 3-4 Epochs
M2_LR              = 2e-5        # 🔧 แนะนำ 2e-5 หรือ 1.5e-5
M2_BATCH_SIZE      = 16          # 🔧 Batch Size
M2_SCHEDULER       = 'cosine'    # 🔧 'cosine', 'linear', 'plateau', หรือ 'none'
M2_PATIENCE        = 2           # 🔧 หยุดทันทีถ้า val loss ไม่ลดลง 2 รอบติด
M2_LOAD_EXISTING   = False       # 🔧 True = เทรนต่อจากเดิม, False = เริ่มใหม่
M3_WARMUP_RATIO    = 0.2        # 🔧 สัดส่วน Warmup (เปลี่ยนชื่อให้เป็น M3_ นำหน้าจะได้สวยๆ)


# 🚀 สั่งรันเทรนเฉพาะ mBERT
ckpt_mbert = train_custom_model(
    model_key=M2_KEY,
    pretrained_name=M2_PRETRAINED_NAME,
    use_fast=M2_USE_FAST,
    epochs=M2_EPOCHS,
    lr=M2_LR,
    batch_size=M2_BATCH_SIZE,
    warmup_ratio=M3_WARMUP_RATIO,
    scheduler_type=M2_SCHEDULER,
    early_stopping_patience=M2_PATIENCE,
    load_existing=M2_LOAD_EXISTING
)


🚀 เริ่มเทรนใหม่ (Fresh Train): [mbert] (bert-base-multilingual-cased)
   ⚙️ Epochs: 10 | LR: 2.00e-05 | Batch: 16 | Scheduler: cosine


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/10 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

/tmp/ipykernel_36209/1814583397.py:78: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 1/10 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

... (2 outputs left out)

Epoch 2/10 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 2/10 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✨ Epoch 2/10 -> Train Loss: 1.0955 | Val Loss: 1.0909 | LR: 2.00e-05
     ⚠️ [Overfit Alert] Val Loss ไม่ลดลงจากจุดดีที่สุด (0.9981) -> จุดนี้ถูกเก็บเป็น Checkpoint ไว้แล้ว

🛑 Early Stopping ทำงาน! (Val Loss ไม่ลดลงติดต่อกัน 2 Epochs)
💡 แนะนำ: คุณสามารถสั่ง ckpt_mgr.rollback_to_best() เพื่อใช้โมเดลตัวที่ดีที่สุดได้เลย
📝 บันทึก Log การเทรนรอบนี้ไว้ที่: ../training_logs_ekman/mbert_run_20261025_193907.json

🎉 เสร็จสิ้นการทำงานของโมเดล [mbert]!

📑 รายการ Checkpoints ทั้งหมดของ [mbert]:
   🏆 [BEST] Epoch 1 | Val Loss: 1.2240 | Train Loss: 1.3781 | LR: 5.00e-05 | เวลา: 2026-10-25 16:04:27
      Epoch 2 | Val Loss: 1.5157 | Train Loss: 1.4698 | LR: 4.85e-05 | เวลา: 2026-10-25 16:10:51
      Epoch 3 | Val Loss: 1.5356 | Train Loss: 1.5153 | LR: 4.42e-05 | เวลา: 2026-10-25 16:17:14
   🏆 [BEST] Epoch 1 | Val Loss: 1.1714 | Train Loss: 1.4282 | LR: 3.11e-05 | เวลา: 2026-10-25 16:28:03
   🏆 [BEST] Epoch 2 | Val Loss: 1.0871 | Train Loss: 1.1875 | LR: 2.33e-05 | เวลา: 2026-10-25 16:34:27
   🏆 [B

In [16]:
# ============================================================
# 🎛️ ปรับแต่ง Hyperparameters สำหรับ โมเดล 2 (mBERT)
# ============================================================
M2_KEY             = 'mbert'
M2_PRETRAINED_NAME = 'bert-base-multilingual-cased'
M2_USE_FAST        = True

M2_EPOCHS          = 10           # 🔧 แนะนำ 3-4 Epochs
M2_LR              = 2e-5        # 🔧 แนะนำ 2e-5 หรือ 1.5e-5
M2_BATCH_SIZE      = 16          # 🔧 Batch Size
M2_SCHEDULER       = 'cosine'    # 🔧 'cosine', 'linear', 'plateau', หรือ 'none'
M2_PATIENCE        = 3           # 🔧 หยุดทันทีถ้า val loss ไม่ลดลง 2 รอบติด
M2_LOAD_EXISTING   = True       # 🔧 True = เทรนต่อจากเดิม, False = เริ่มใหม่
M3_WARMUP_RATIO    = 0.2        # 🔧 สัดส่วน Warmup (เปลี่ยนชื่อให้เป็น M3_ นำหน้าจะได้สวยๆ)


# 🚀 สั่งรันเทรนเฉพาะ mBERT
ckpt_mbert = train_custom_model(
    model_key=M2_KEY,
    pretrained_name=M2_PRETRAINED_NAME,
    use_fast=M2_USE_FAST,
    epochs=M2_EPOCHS,
    lr=M2_LR,
    batch_size=M2_BATCH_SIZE,
    warmup_ratio=M3_WARMUP_RATIO,
    scheduler_type=M2_SCHEDULER,
    early_stopping_patience=M2_PATIENCE,
    load_existing=M2_LOAD_EXISTING
)


🔄 โหลดโมเดลเดิมมาเทรนต่อ (Continue): [mbert] (bert-base-multilingual-cased)
   ⚙️ Epochs: 10 | LR: 2.00e-05 | Batch: 16 | Scheduler: cosine
📂 กำลังโหลดโมเดลเดิมจาก: ../saved_models_ekman_finetune/mbert_model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch 1/10 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 1/10 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✨ Epoch 1/10 -> Train Loss: 0.9398 | Val Loss: 1.0079 | LR: 1.00e-05
     ⚠️ [Overfit Alert] Val Loss ไม่ลดลงจากจุดดีที่สุด (0.9981) -> จุดนี้ถูกเก็บเป็น Checkpoint ไว้แล้ว


... (4 outputs left out)

Epoch 3/10 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 3/10 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✨ Epoch 3/10 -> Train Loss: 0.9296 | Val Loss: 1.0047 | LR: 1.92e-05
     ⚠️ [Overfit Alert] Val Loss ไม่ลดลงจากจุดดีที่สุด (0.9981) -> จุดนี้ถูกเก็บเป็น Checkpoint ไว้แล้ว

🛑 Early Stopping ทำงาน! (Val Loss ไม่ลดลงติดต่อกัน 3 Epochs)
💡 แนะนำ: คุณสามารถสั่ง ckpt_mgr.rollback_to_best() เพื่อใช้โมเดลตัวที่ดีที่สุดได้เลย
📝 บันทึก Log การเทรนรอบนี้ไว้ที่: ../training_logs_ekman/mbert_run_20261025_195946.json

🎉 เสร็จสิ้นการทำงานของโมเดล [mbert]!

📑 รายการ Checkpoints ทั้งหมดของ [mbert]:
   🏆 [BEST] Epoch 1 | Val Loss: 1.2240 | Train Loss: 1.3781 | LR: 5.00e-05 | เวลา: 2026-10-25 16:04:27
      Epoch 2 | Val Loss: 1.5157 | Train Loss: 1.4698 | LR: 4.85e-05 | เวลา: 2026-10-25 16:10:51
      Epoch 3 | Val Loss: 1.5356 | Train Loss: 1.5153 | LR: 4.42e-05 | เวลา: 2026-10-25 16:17:14
   🏆 [BEST] Epoch 1 | Val Loss: 1.1714 | Train Loss: 1.4282 | LR: 3.11e-05 | เวลา: 2026-10-25 16:28:03
   🏆 [BEST] Epoch 2 | Val Loss: 1.0871 | Train Loss: 1.1875 | LR: 2.33e-05 | เวลา: 2026-10-25 16:34:27
   🏆 [B

In [17]:
# ============================================================
# 🎛️ ปรับแต่ง Hyperparameters สำหรับ โมเดล 2 (mBERT)
# ============================================================
M2_KEY             = 'mbert'
M2_PRETRAINED_NAME = 'bert-base-multilingual-cased'
M2_USE_FAST        = True

M2_EPOCHS          = 10           # 🔧 แนะนำ 3-4 Epochs
M2_LR              = 1.8e-5        # 🔧 แนะนำ 2e-5 หรือ 1.5e-5
M2_BATCH_SIZE      = 16          # 🔧 Batch Size
M2_SCHEDULER       = 'cosine'    # 🔧 'cosine', 'linear', 'plateau', หรือ 'none'
M2_PATIENCE        = 3           # 🔧 หยุดทันทีถ้า val loss ไม่ลดลง 2 รอบติด
M2_LOAD_EXISTING   = True       # 🔧 True = เทรนต่อจากเดิม, False = เริ่มใหม่
M3_WARMUP_RATIO    = 0.1        # 🔧 สัดส่วน Warmup (เปลี่ยนชื่อให้เป็น M3_ นำหน้าจะได้สวยๆ)


# 🚀 สั่งรันเทรนเฉพาะ mBERT
ckpt_mbert = train_custom_model(
    model_key=M2_KEY,
    pretrained_name=M2_PRETRAINED_NAME,
    use_fast=M2_USE_FAST,
    epochs=M2_EPOCHS,
    lr=M2_LR,
    batch_size=M2_BATCH_SIZE,
    warmup_ratio=M3_WARMUP_RATIO,
    scheduler_type=M2_SCHEDULER,
    early_stopping_patience=M2_PATIENCE,
    load_existing=M2_LOAD_EXISTING
)


🔄 โหลดโมเดลเดิมมาเทรนต่อ (Continue): [mbert] (bert-base-multilingual-cased)
   ⚙️ Epochs: 10 | LR: 1.80e-05 | Batch: 16 | Scheduler: cosine
📂 กำลังโหลดโมเดลเดิมจาก: ../saved_models_ekman_finetune/mbert_model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch 1/10 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 1/10 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✨ Epoch 1/10 -> Train Loss: 0.9516 | Val Loss: 1.0751 | LR: 1.80e-05
     ⚠️ [Overfit Alert] Val Loss ไม่ลดลงจากจุดดีที่สุด (0.9981) -> จุดนี้ถูกเก็บเป็น Checkpoint ไว้แล้ว


... (4 outputs left out)

Epoch 3/10 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 3/10 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✨ Epoch 3/10 -> Train Loss: 0.8882 | Val Loss: 1.0308 | LR: 1.59e-05
     ⚠️ [Overfit Alert] Val Loss ไม่ลดลงจากจุดดีที่สุด (0.9981) -> จุดนี้ถูกเก็บเป็น Checkpoint ไว้แล้ว

🛑 Early Stopping ทำงาน! (Val Loss ไม่ลดลงติดต่อกัน 3 Epochs)
💡 แนะนำ: คุณสามารถสั่ง ckpt_mgr.rollback_to_best() เพื่อใช้โมเดลตัวที่ดีที่สุดได้เลย
📝 บันทึก Log การเทรนรอบนี้ไว้ที่: ../training_logs_ekman/mbert_run_20261025_202114.json

🎉 เสร็จสิ้นการทำงานของโมเดล [mbert]!

📑 รายการ Checkpoints ทั้งหมดของ [mbert]:
   🏆 [BEST] Epoch 1 | Val Loss: 1.2240 | Train Loss: 1.3781 | LR: 5.00e-05 | เวลา: 2026-10-25 16:04:27
      Epoch 2 | Val Loss: 1.5157 | Train Loss: 1.4698 | LR: 4.85e-05 | เวลา: 2026-10-25 16:10:51
      Epoch 3 | Val Loss: 1.5356 | Train Loss: 1.5153 | LR: 4.42e-05 | เวลา: 2026-10-25 16:17:14
   🏆 [BEST] Epoch 1 | Val Loss: 1.1714 | Train Loss: 1.4282 | LR: 3.11e-05 | เวลา: 2026-10-25 16:28:03
   🏆 [BEST] Epoch 2 | Val Loss: 1.0871 | Train Loss: 1.1875 | LR: 2.33e-05 | เวลา: 2026-10-25 16:34:27
   🏆 [B

In [18]:
ckpt_mbert.rollback_to_best()

⏪ กำลัง Rollback โมเดลหลัก [mbert] ไปใช้ Epoch 3...
✅ Rollback สำเร็จ! โมเดลหลักที่ ../saved_models_ekman_finetune/mbert_model กลายเป็นเวอร์ชัน Epoch 3 แล้ว


True

In [ ]:
#rollback กลับไปแต่โมเดล ที่ดีที่สุดโดนทับไปแล้วเลยเริ่มเทรนใหม่ดีกว่า

In [9]:
# ============================================================
# 🎛️ ปรับแต่ง Hyperparameters สำหรับ โมเดล 2 (mBERT)
# ============================================================
M2_KEY             = 'mbert'
M2_PRETRAINED_NAME = 'bert-base-multilingual-cased'
M2_USE_FAST        = True

M2_EPOCHS          = 10           # 🔧 แนะนำ 3-4 Epochs
M2_LR              = 2.5e-5        # 🔧 แนะนำ 2e-5 หรือ 1.5e-5
M2_BATCH_SIZE      = 16          # 🔧 Batch Size
M2_SCHEDULER       = 'cosine'    # 🔧 'cosine', 'linear', 'plateau', หรือ 'none'
M2_PATIENCE        = 3           # 🔧 หยุดทันทีถ้า val loss ไม่ลดลง 2 รอบติด
M2_LOAD_EXISTING   = False       # 🔧 True = เทรนต่อจากเดิม, False = เริ่มใหม่
M3_WARMUP_RATIO    = 0.2        # 🔧 สัดส่วน Warmup (เปลี่ยนชื่อให้เป็น M3_ นำหน้าจะได้สวยๆ)


# 🚀 สั่งรันเทรนเฉพาะ mBERT
ckpt_mbert = train_custom_model(
    model_key=M2_KEY,
    pretrained_name=M2_PRETRAINED_NAME,
    use_fast=M2_USE_FAST,
    epochs=M2_EPOCHS,
    lr=M2_LR,
    batch_size=M2_BATCH_SIZE,
    warmup_ratio=M3_WARMUP_RATIO,
    scheduler_type=M2_SCHEDULER,
    early_stopping_patience=M2_PATIENCE,
    load_existing=M2_LOAD_EXISTING
)


🚀 เริ่มเทรนใหม่ (Fresh Train): [mbert] (bert-base-multilingual-cased)
   ⚙️ Epochs: 10 | LR: 2.50e-05 | Batch: 16 | Scheduler: cosine


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/10 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 1/10 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (34 outputs left out)

Epoch 9/10 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 9/10 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✨ Epoch 9/10 -> Train Loss: 0.6128 | Val Loss: 1.2685 | LR: 9.52e-07
     ⚠️ [Overfit Alert] Val Loss ไม่ลดลงจากจุดดีที่สุด (0.9331) -> จุดนี้ถูกเก็บเป็น Checkpoint ไว้แล้ว

🛑 Early Stopping ทำงาน! (Val Loss ไม่ลดลงติดต่อกัน 3 Epochs)
💡 แนะนำ: คุณสามารถสั่ง ckpt_mgr.rollback_to_best() เพื่อใช้โมเดลตัวที่ดีที่สุดได้เลย
📝 บันทึก Log การเทรนรอบนี้ไว้ที่: ../training_logs_ekman/mbert_run_20261025_212911.json

🎉 เสร็จสิ้นการทำงานของโมเดล [mbert]!

📑 รายการ Checkpoints ทั้งหมดของ [mbert]:
   🏆 [BEST] Epoch 1 | Val Loss: 1.1031 | Train Loss: 1.3986 | LR: 1.25e-05 | เวลา: 2026-10-25 20:38:15
   🏆 [BEST] Epoch 2 | Val Loss: 1.0331 | Train Loss: 1.1160 | LR: 2.50e-05 | เวลา: 2026-10-25 20:44:39
   🏆 [BEST] Epoch 3 | Val Loss: 1.0121 | Train Loss: 1.0456 | LR: 2.40e-05 | เวลา: 2026-10-25 20:51:02
   🏆 [BEST] Epoch 4 | Val Loss: 0.9912 | Train Loss: 0.9682 | LR: 2.13e-05 | เวลา: 2026-10-25 20:57:25
      Epoch 5 | Val Loss: 1.0582 | Train Loss: 0.9024 | LR: 1.73e-05 | เวลา: 2026-10-25 21:03:46
 

In [10]:
ckpt_mbert.rollback_to_best()

⏪ กำลัง Rollback โมเดลหลัก [mbert] ไปใช้ Epoch 6...
✅ Rollback สำเร็จ! โมเดลหลักที่ ../saved_models_ekman_finetune/mbert_model กลายเป็นเวอร์ชัน Epoch 6 แล้ว


True

## 🚀 โมเดล 3: XLM-RoBERTa

> **คุณสามารถปรับตั้งค่า `EPOCHS`, `LR`, `BATCH_SIZE`, `SCHEDULER` ได้ที่เซลล์นี้ แล้วกดรันเฉพาะเซลล์นี้ได้เลยครับ**

In [8]:
# ============================================================
# 🎛️ ปรับแต่ง Hyperparameters สำหรับ โมเดล 3 (XLM-RoBERTa)
# ============================================================
M3_KEY             = 'xlm_roberta'
M3_PRETRAINED_NAME = 'xlm-roberta-base'
M3_USE_FAST        = True

M3_EPOCHS          = 5           # 🔧 จำนวน Epochs
M3_LR              = 5e-5      # 🔧 แนะนำ 1.5e-5 หรือ 2e-5 สำหรับ RoBERTa
M3_BATCH_SIZE      = 16          # 🔧 Batch Size
M3_SCHEDULER       = 'cosine'    # 🔧 'cosine', 'linear', 'plateau', หรือ 'none'
M3_PATIENCE        = 2           # 🔧 Early stopping patience
M3_LOAD_EXISTING   = False       # 🔧 True = เทรนต่อจากเดิม, False = เริ่มใหม่

# 🚀 สั่งรันเทรนเฉพาะ XLM-RoBERTa
ckpt_xlm = train_custom_model(
    model_key=M3_KEY,
    pretrained_name=M3_PRETRAINED_NAME,
    use_fast=M3_USE_FAST,
    epochs=M3_EPOCHS,
    lr=M3_LR,
    batch_size=M3_BATCH_SIZE,
    scheduler_type=M3_SCHEDULER,
    early_stopping_patience=M3_PATIENCE,
    load_existing=M3_LOAD_EXISTING
)


🚀 เริ่มเทรนใหม่ (Fresh Train): [xlm_roberta] (xlm-roberta-base)
   ⚙️ Epochs: 5 | LR: 5.00e-05 | Batch: 16 | Scheduler: cosine


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/5 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

/tmp/ipykernel_36209/1814583397.py:78: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 1/5 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

... (18 outputs left out)

Epoch 5/5 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 5/5 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✨ Epoch 5/5 -> Train Loss: 1.5080 | Val Loss: 1.5154 | LR: 0.00e+00
     ⚠️ [Overfit Alert] Val Loss ไม่ลดลงจากจุดดีที่สุด (1.5145) -> จุดนี้ถูกเก็บเป็น Checkpoint ไว้แล้ว
📝 บันทึก Log การเทรนรอบนี้ไว้ที่: ../training_logs_ekman/xlm_roberta_run_20261025_174422.json

🎉 เสร็จสิ้นการทำงานของโมเดล [xlm_roberta]!

📑 รายการ Checkpoints ทั้งหมดของ [xlm_roberta]:
   🏆 [BEST] Epoch 1 | Val Loss: 1.5298 | Train Loss: 1.5107 | LR: 4.85e-05 | เวลา: 2026-10-25 17:11:27
   🏆 [BEST] Epoch 2 | Val Loss: 1.5190 | Train Loss: 1.5184 | LR: 3.75e-05 | เวลา: 2026-10-25 17:19:42
   🏆 [BEST] Epoch 3 | Val Loss: 1.5180 | Train Loss: 1.5143 | LR: 2.07e-05 | เวลา: 2026-10-25 17:27:55
   🏆 [BEST] Epoch 4 | Val Loss: 1.5145 | Train Loss: 1.5094 | LR: 5.85e-06 | เวลา: 2026-10-25 17:36:07
      Epoch 5 | Val Loss: 1.5154 | Train Loss: 1.5080 | LR: 0.00e+00 | เวลา: 2026-10-25 17:44:19


In [10]:
ckpt_xlm.rollback_to_best()

⏪ กำลัง Rollback โมเดลหลัก [xlm_roberta] ไปใช้ Epoch 4...
✅ Rollback สำเร็จ! โมเดลหลักที่ ../saved_models_ekman_finetune/xlm_roberta_model กลายเป็นเวอร์ชัน Epoch 4 แล้ว


True

In [ ]:
# ============================================================
# 🎛️ ปรับแต่ง Hyperparameters สำหรับ โมเดล 3 (XLM-RoBERTa)
# ============================================================
M3_KEY             = 'xlm_roberta'
M3_PRETRAINED_NAME = 'xlm-roberta-base'
M3_USE_FAST        = True

M3_EPOCHS          = 5           # 🔧 จำนวน Epochs
M3_LR              = 4e-5      # 🔧 แนะนำ 1.5e-5 หรือ 2e-5 สำหรับ RoBERTa
M3_BATCH_SIZE      = 16          # 🔧 Batch Size
M3_SCHEDULER       = 'linear'    # 🔧 'cosine', 'linear', 'plateau', หรือ 'none'
M3_PATIENCE        = 2           # 🔧 Early stopping patience
M3_LOAD_EXISTING   = True       # 🔧 True = เทรนต่อจากเดิม, False = เริ่มใหม่
M3_WARMUP_RATIO    = 0.3        # 🔧 สัดส่วน Warmup (เปลี่ยนชื่อให้เป็น M3_ นำหน้าจะได้สวยๆ)
# 🚀 สั่งรันเทรนเฉพาะ XLM-RoBERTa
ckpt_xlm = train_custom_model(
    model_key=M3_KEY,
    pretrained_name=M3_PRETRAINED_NAME,
    use_fast=M3_USE_FAST,
    epochs=M3_EPOCHS,
    lr=M3_LR,
    batch_size=M3_BATCH_SIZE,
    warmup_ratio=M3_WARMUP_RATIO,
    scheduler_type=M3_SCHEDULER,
    early_stopping_patience=M3_PATIENCE,
    load_existing=M3_LOAD_EXISTING
)


🔄 โหลดโมเดลเดิมมาเทรนต่อ (Continue): [xlm_roberta] (xlm-roberta-base)
   ⚙️ Epochs: 5 | LR: 4.00e-05 | Batch: 16 | Scheduler: linear
📂 กำลังโหลดโมเดลเดิมจาก: ../saved_models_ekman_finetune/xlm_roberta_model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch 1/5 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 1/5 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✨ Epoch 1/5 -> Train Loss: 1.5107 | Val Loss: 1.5233 | LR: 2.67e-05
     ⚠️ [Overfit Alert] Val Loss ไม่ลดลงจากจุดดีที่สุด (1.5145) -> จุดนี้ถูกเก็บเป็น Checkpoint ไว้แล้ว


Epoch 2/5 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 2/5 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✨ Epoch 2/5 -> Train Loss: 1.5138 | Val Loss: 1.5167 | LR: 3.43e-05
     ⚠️ [Overfit Alert] Val Loss ไม่ลดลงจากจุดดีที่สุด (1.5145) -> จุดนี้ถูกเก็บเป็น Checkpoint ไว้แล้ว

🛑 Early Stopping ทำงาน! (Val Loss ไม่ลดลงติดต่อกัน 2 Epochs)
💡 แนะนำ: คุณสามารถสั่ง ckpt_mgr.rollback_to_best() เพื่อใช้โมเดลตัวที่ดีที่สุดได้เลย
📝 บันทึก Log การเทรนรอบนี้ไว้ที่: ../training_logs_ekman/xlm_roberta_run_20261025_181103.json

🎉 เสร็จสิ้นการทำงานของโมเดล [xlm_roberta]!

📑 รายการ Checkpoints ทั้งหมดของ [xlm_roberta]:
   🏆 [BEST] Epoch 1 | Val Loss: 1.5298 | Train Loss: 1.5107 | LR: 4.85e-05 | เวลา: 2026-10-25 17:11:27
   🏆 [BEST] Epoch 2 | Val Loss: 1.5190 | Train Loss: 1.5184 | LR: 3.75e-05 | เวลา: 2026-10-25 17:19:42
   🏆 [BEST] Epoch 3 | Val Loss: 1.5180 | Train Loss: 1.5143 | LR: 2.07e-05 | เวลา: 2026-10-25 17:27:55
   🏆 [BEST] Epoch 4 | Val Loss: 1.5145 | Train Loss: 1.5094 | LR: 5.85e-06 | เวลา: 2026-10-25 17:36:07
      Epoch 5 | Val Loss: 1.5154 | Train Loss: 1.5080 | LR: 0.00e+00 | เวลา: 2026

In [12]:
ckpt_xlm.rollback_to_best()

⏪ กำลัง Rollback โมเดลหลัก [xlm_roberta] ไปใช้ Epoch 4...
✅ Rollback สำเร็จ! โมเดลหลักที่ ../saved_models_ekman_finetune/xlm_roberta_model กลายเป็นเวอร์ชัน Epoch 4 แล้ว


True

In [13]:
# ============================================================
# 🎛️ ปรับแต่ง Hyperparameters สำหรับ โมเดล 3 (XLM-RoBERTa)
# ============================================================
M3_KEY             = 'xlm_roberta'
M3_PRETRAINED_NAME = 'xlm-roberta-base'
M3_USE_FAST        = True

M3_EPOCHS          = 10           # 🔧 จำนวน Epochs
M3_LR              = 2e-5      # 🔧 แนะนำ 1.5e-5 หรือ 2e-5 สำหรับ RoBERTa
M3_BATCH_SIZE      = 16          # 🔧 Batch Size
M3_SCHEDULER       = 'cosine'    # 🔧 'cosine', 'linear', 'plateau', หรือ 'none'
M3_PATIENCE        = 2           # 🔧 Early stopping patience
M3_LOAD_EXISTING   = False       # 🔧 True = เทรนต่อจากเดิม, False = เริ่มใหม่
M3_WARMUP_RATIO    = 0.2        # 🔧 สัดส่วน Warmup (เปลี่ยนชื่อให้เป็น M3_ นำหน้าจะได้สวยๆ)
# 🚀 สั่งรันเทรนเฉพาะ XLM-RoBERTa
ckpt_xlm = train_custom_model(
    model_key=M3_KEY,
    pretrained_name=M3_PRETRAINED_NAME,
    use_fast=M3_USE_FAST,
    epochs=M3_EPOCHS,
    lr=M3_LR,
    batch_size=M3_BATCH_SIZE,
    warmup_ratio=M3_WARMUP_RATIO,
    scheduler_type=M3_SCHEDULER,
    early_stopping_patience=M3_PATIENCE,
    load_existing=M3_LOAD_EXISTING
)


🚀 เริ่มเทรนใหม่ (Fresh Train): [xlm_roberta] (xlm-roberta-base)
   ⚙️ Epochs: 10 | LR: 2.00e-05 | Batch: 16 | Scheduler: cosine


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/10 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 1/10 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (21 outputs left out)

Epoch 6/10 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 6/10 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✨ Epoch 6/10 -> Train Loss: 0.6710 | Val Loss: 0.9480 | LR: 1.00e-05
     ⚠️ [Overfit Alert] Val Loss ไม่ลดลงจากจุดดีที่สุด (0.8436) -> จุดนี้ถูกเก็บเป็น Checkpoint ไว้แล้ว

🛑 Early Stopping ทำงาน! (Val Loss ไม่ลดลงติดต่อกัน 2 Epochs)
💡 แนะนำ: คุณสามารถสั่ง ckpt_mgr.rollback_to_best() เพื่อใช้โมเดลตัวที่ดีที่สุดได้เลย
📝 บันทึก Log การเทรนรอบนี้ไว้ที่: ../training_logs_ekman/xlm_roberta_run_20261025_191628.json

🎉 เสร็จสิ้นการทำงานของโมเดล [xlm_roberta]!

📑 รายการ Checkpoints ทั้งหมดของ [xlm_roberta]:
   🏆 [BEST] Epoch 1 | Val Loss: 1.5298 | Train Loss: 1.5107 | LR: 4.85e-05 | เวลา: 2026-10-25 17:11:27
   🏆 [BEST] Epoch 2 | Val Loss: 1.5190 | Train Loss: 1.5184 | LR: 3.75e-05 | เวลา: 2026-10-25 17:19:42
   🏆 [BEST] Epoch 3 | Val Loss: 1.5180 | Train Loss: 1.5143 | LR: 2.07e-05 | เวลา: 2026-10-25 17:27:55
   🏆 [BEST] Epoch 4 | Val Loss: 1.5145 | Train Loss: 1.5094 | LR: 5.85e-06 | เวลา: 2026-10-25 17:36:07
      Epoch 5 | Val Loss: 1.5154 | Train Loss: 1.5080 | LR: 0.00e+00 | เวลา: 202

In [14]:
ckpt_xlm.rollback_to_best()

⏪ กำลัง Rollback โมเดลหลัก [xlm_roberta] ไปใช้ Epoch 4...
✅ Rollback สำเร็จ! โมเดลหลักที่ ../saved_models_ekman_finetune/xlm_roberta_model กลายเป็นเวอร์ชัน Epoch 4 แล้ว


True

## ⏪ Step 7: ระบบ Rollback ย้อนคืน Checkpoint กรณี Overfit

> หากคุณเทรนไปหลาย Epoch แล้วพบว่า Epoch ท้ายๆ Val Loss ดีดขึ้น (Overfit) 
> คุณสามารถสั่ง **Rollback** ย้อนกลับไปใช้ Epoch ที่ดีที่สุด หรือระบุ Epoch ที่ต้องการได้ที่เซลล์นี้ครับ

In [ ]:
# ============================================================
# # ⏪ ตัวอย่างการ Rollback โมเดล
# # ============================================================
# TARGET_MODEL = 'mbert'  # 🔧 เลือกโมเดลที่ต้องการ Rollback ('wangchan', 'mbert', 'xlm_roberta')

# manager = CheckpointManager(model_key=TARGET_MODEL, base_dir=BASE_SAVE_DIR)
# manager.list_checkpoints()

# # --- ตัวเลือกที่ 1: Rollback กลับไปใช้ Best Model อัตโนมัติ ---
# manager.rollback_to_best()

# # --- ตัวเลือกที่ 2: Rollback ไปยัง Epoch ที่เจาะจง (ปลดคอมเมนต์ถ้าต้องการ) ---
# # manager.rollback_to_epoch(target_epoch=2)

## 🔮 Step 8: สกัด 21-dim Probability Features (7 + 7 + 7) จากทั้ง 3 โมเดล

In [11]:
def extract_probs_safe(model_key, texts, name, batch_size=32):
    """โหลดโมเดลที่เซฟไว้มาสร้าง Probability และคืน VRAM ทันที"""
    model_dir = os.path.join(BASE_SAVE_DIR, f"{model_key}_model")
    tok_dir   = os.path.join(BASE_SAVE_DIR, f"{model_key}_tokenizer")
    
    if not os.path.exists(model_dir):
        raise FileNotFoundError(f"❌ ไม่พบโฟลเดอร์โมเดล {model_dir}! กรุณารันเซลล์เทรนโมเดล {model_key} ก่อน")
        
    print(f"\n🔮 กำลังสกัด Feature จาก {name} (ที่อยู่: {model_dir})...")
    tok = AutoTokenizer.from_pretrained(tok_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
    model.eval()
    
    ds = ThaiEmotionDataset(texts, np.zeros((len(texts), NUM_LABELS)), tok)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    probs = []
    
    with torch.no_grad():
        for b in tqdm(loader, desc=f"Extracting {name}"):
            input_ids = b['input_ids'].to(device)
            mask = b['attention_mask'].to(device)
            with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
                out = model(input_ids, attention_mask=mask)
                probs.append(torch.sigmoid(out.logits).cpu().numpy())
                
    # ดีดโมเดลออกและเคลียร์ VRAM ทันที
    model.to('cpu')
    del model, tok, ds, loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return np.concatenate(probs, axis=0)

# สกัด Features จากทั้ง 3 โมเดลทีละตัว (ปลอดภัย VRAM 8GB)
train_w = extract_probs_safe('wangchan', X_train, 'WangchanBERTa (Train)')
test_w  = extract_probs_safe('wangchan', X_test,  'WangchanBERTa (Test)')

train_m = extract_probs_safe('mbert', X_train, 'mBERT (Train)')
test_m  = extract_probs_safe('mbert', X_test,  'mBERT (Test)')

train_x = extract_probs_safe('xlm_roberta', X_train, 'XLM-RoBERTa (Train)')
test_x  = extract_probs_safe('xlm_roberta', X_test,  'XLM-RoBERTa (Test)')

# รวมเป็น Stacked Probabilities 21 มิติ
X_train_meta = np.concatenate([train_w, train_m, train_x], axis=1)
X_test_meta  = np.concatenate([test_w, test_m, test_x], axis=1)

print(f"\n✅ รวบรวม 21-dim Probability Features สำเร็จ! Shape: Train = {X_train_meta.shape} | Test = {X_test_meta.shape}")


🔮 กำลังสกัด Feature จาก WangchanBERTa (Train) (ที่อยู่: ../saved_models_ekman_finetune/wangchan_model)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Extracting WangchanBERTa (Train):   0%|          | 0/1357 [00:00<?, ?it/s]


🔮 กำลังสกัด Feature จาก WangchanBERTa (Test) (ที่อยู่: ../saved_models_ekman_finetune/wangchan_model)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Extracting WangchanBERTa (Test):   0%|          | 0/161 [00:00<?, ?it/s]

... (9 outputs left out)


🔮 กำลังสกัด Feature จาก XLM-RoBERTa (Test) (ที่อยู่: ../saved_models_ekman_finetune/xlm_roberta_model)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Extracting XLM-RoBERTa (Test):   0%|          | 0/161 [00:00<?, ?it/s]


✅ รวบรวม 21-dim Probability Features สำเร็จ! Shape: Train = (43410, 21) | Test = (5131, 21)


## 🌲 Step 9: Level 2 Class Weighting: เทรน Random Forest Meta-Classifier

In [12]:
# ปรับจูน Class Weight แยกอิสระตามแต่ละคลาสใน Random Forest
RF_CLASS_WEIGHTS = [
    'balanced',        # [0] Joy
    'balanced',        # [1] Anger
    {0: 1, 1: 3},      # [2] Sadness  (boost 3x)
    'balanced',        # [3] Surprise
    {0: 1, 1: 5},      # [4] Fear     (boost 5x)
    {0: 1, 1: 5},      # [5] Disgust  (boost 5x)
    'balanced',        # [6] Neutral
]

print('🌲 กำลังเทรน Random Forest Meta-Classifier ด้วย Custom Class Weighting...')
fitted_estimators = []
for i in range(NUM_LABELS):
    rf = RandomForestClassifier(
        n_estimators=200,
        class_weight=RF_CLASS_WEIGHTS[i],
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train_meta, Y_train[:, i])
    fitted_estimators.append(rf)
    print(f'  ✅ Fitted estimator for [{i}] {EKMAN_LABELS[i]} (weight: {RF_CLASS_WEIGHTS[i]})')

rf_path = os.path.join(BASE_SAVE_DIR, 'random_forest_ekman_weighted.joblib')
labels_path = os.path.join(BASE_SAVE_DIR, 'label_classes.json')

joblib.dump(fitted_estimators, rf_path)
with open(labels_path, 'w', encoding='utf-8') as f:
    json.dump(EKMAN_LABELS, f, ensure_ascii=False, indent=2)

print(f"\n💾 บันทึก Random Forest และ Label Classes สำเร็จเรียบร้อยที่: {BASE_SAVE_DIR}")

🌲 กำลังเทรน Random Forest Meta-Classifier ด้วย Custom Class Weighting...
  ✅ Fitted estimator for [0] Joy (weight: balanced)
  ✅ Fitted estimator for [1] Anger (weight: balanced)
  ✅ Fitted estimator for [2] Sadness (weight: {0: 1, 1: 3})
  ✅ Fitted estimator for [3] Surprise (weight: balanced)
  ✅ Fitted estimator for [4] Fear (weight: {0: 1, 1: 5})
  ✅ Fitted estimator for [5] Disgust (weight: {0: 1, 1: 5})
  ✅ Fitted estimator for [6] Neutral (weight: balanced)

💾 บันทึก Random Forest และ Label Classes สำเร็จเรียบร้อยที่: ../saved_models_ekman_finetune


## 📈 Step 10: ประเมินผลลัพธ์ & บันทึกรายงานการทดลอง

In [13]:
Y_pred_cols = [est.predict(X_test_meta) for est in fitted_estimators]
Y_pred = np.stack(Y_pred_cols, axis=1)

micro_f1 = f1_score(Y_test, Y_pred, average='micro', zero_division=0)
macro_f1 = f1_score(Y_test, Y_pred, average='macro', zero_division=0)
weighted_f1 = f1_score(Y_test, Y_pred, average='weighted', zero_division=0)
h_loss = hamming_loss(Y_test, Y_pred)

print('=' * 65)
print('📊 สรุปผลการประเมิน Fine-Tuned Ekman 7 Ensemble (Full Test Set)')
print('=' * 65)
print(f'  Micro F1     : {micro_f1:.4f}')
print(f'  Macro F1     : {macro_f1:.4f}')
print(f'  Weighted F1  : {weighted_f1:.4f}')
print(f'  Hamming Loss : {h_loss:.4f}')
print('=' * 65)
print('\nClassification Report:')
print(classification_report(Y_test, Y_pred, target_names=EKMAN_LABELS, zero_division=0))

if save_eval_report is not None:
    save_eval_report(
        y_true=Y_test,
        y_pred=Y_pred,
        target_names=EKMAN_LABELS,
        model_name='Fine-Tuned Ekman 7 + 2-Level Class Weighting (Modular)',
        hyperparams={
            'Save Dir': BASE_SAVE_DIR,
            'Taxonomy': 'Ekman 7 Emotions',
            'Base Models': 'WangchanBERTa, mBERT, XLM-RoBERTa (Modular Checkpoints)',
            'Loss Weighting': 'BCE pos_weight + Boost (Fear x3.0, Disgust x3.0, Sadness x2.5)',
            'Level 2 RF': 'RandomForest (Fear/Disgust {0:1,1:5}, Sadness {0:1,1:3})'
        },
        notes='เทรนผ่าน Modular Notebook พร้อมระบบ Best Checkpoint, LR Scheduler, Logs และ Rollback',
        output_dir=RESULTS_DIR
    )
    print('🎉 บันทึกรายงานการประเมินลงโฟลเดอร์ results/ และ SUMMARY.md สำเร็จเรียบร้อย!')
else:
    print('ℹ️ ผลการประเมินแสดงผลข้างต้นเรียบร้อยแล้ว')

📊 สรุปผลการประเมิน Fine-Tuned Ekman 7 Ensemble (Full Test Set)
  Micro F1     : 0.6004
  Macro F1     : 0.4988
  Weighted F1  : 0.5998
  Hamming Loss : 0.1267

Classification Report:
              precision    recall  f1-score   support

         Joy       0.74      0.73      0.74      2008
       Anger       0.45      0.47      0.46       687
     Sadness       0.53      0.45      0.49       359
    Surprise       0.49      0.55      0.52       645
        Fear       0.45      0.41      0.43        88
     Disgust       0.50      0.20      0.28       117
     Neutral       0.54      0.63      0.58      1670

   micro avg       0.59      0.61      0.60      5574
   macro avg       0.53      0.49      0.50      5574
weighted avg       0.59      0.61      0.60      5574
 samples avg       0.59      0.63      0.59      5574

✅ Saved evaluation report to markdown file: ../results/eval_2026-10-25_21-45-41.md
📊 Updated experiment leaderboard: ../results/SUMMARY.md
🎉 บันทึกรายงานการประเมินลงโ